# STL-Zerlegung einfach erklärt mit einem realen Datensatz

Dieses Notebook erklärt die **STL-Methode** und wendet sie auf einen einfachen Zeitreihen-Datensatz an: monatliche Zahlen internationaler Flugpassagiere.

STL steht für:

- **S**aisonal
- **T**rend
- Zerlegung mithilfe von **L**OESS

Das Ziel ist, eine Zeitreihe in drei verständliche Bestandteile zu zerlegen:

\[
y_t = T_t + S_t + R_t
\]

wobei gilt:

- \(y_t\) = beobachteter Wert zum Zeitpunkt \(t\)
- \(T_t\) = langfristiger Trend
- \(S_t\) = wiederkehrendes saisonales Muster
- \(R_t\) = Residuum/Rauschen, das nach dem Entfernen von Trend und Saisonalität übrig bleibt

Das ist für geschäftliche Prognosen nützlich, weil wir dadurch besser verstehen, ob Veränderungen durch Wachstum, Saisonalität oder unregelmäßige Effekte verursacht werden.


## 1. Geschäftsproblem

Stellen Sie sich vor, Sie arbeiten für eine Fluggesellschaft und möchten die Passagiernachfrage verstehen.

Das Unternehmen hat monatliche Passagierzahlen und möchte folgende Fragen beantworten:

> Steigen die Passagierzahlen im Laufe der Zeit?  
> Gibt es ein wiederkehrendes jährliches saisonales Muster?  
> Gibt es ungewöhnliche Monate, die weder dem normalen Trend noch der üblichen Saisonalität folgen?

Das ist ein guter Anwendungsfall für STL, weil die Nachfrage im Flugverkehr normalerweise beides enthält:

- einen langfristigen Wachstumstrend
- jährliche Saisonalität, zum Beispiel eine höhere Reisenachfrage im Sommer


In [ ]:
# Fehlende Pakete bei Bedarf installieren:
# !pip install pandas matplotlib statsmodels

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.datasets import get_rdataset

plt.rcParams["figure.figsize"] = (12, 5)


## 2. Datensatz laden

Wir verwenden den klassischen **AirPassengers**-Datensatz.

Er enthält monatliche Gesamtzahlen internationaler Flugpassagiere von 1949 bis 1960.

Der Datensatz ist klein, sauber und leicht verständlich. Dadurch eignet er sich ideal, um STL zu lernen.


In [ ]:
# AirPassengers-Datensatz aus dem statsmodels/R-Datensatz-Repository laden
# Falls dies wegen fehlendem Internetzugang fehlschlägt, verwenden Sie die Fallback-Zelle unten.

data = get_rdataset("AirPassengers", package="datasets").data

data.head()


In [ ]:
# Den Datensatz in eine korrekte monatliche Zeitreihe umwandeln

ts = data.copy()
ts["date"] = pd.date_range(start="1949-01-01", periods=len(ts), freq="MS")
ts = ts.set_index("date")
ts = ts.rename(columns={"value": "passengers"})

ts.head()


### Optionaler Ausweich-Datensatz

Falls der vorherige Ladeschritt nicht funktioniert, weil Ihre Umgebung keinen Internetzugang hat, führen Sie stattdessen diese Ausweich-Zelle aus.


In [ ]:
# Ausweichlösung: eingebettete AirPassengers-Daten
# Diese Zelle nur ausführen, wenn get_rdataset oben fehlgeschlagen ist.

# values = [
#     112,118,132,129,121,135,148,148,136,119,104,118,
#     115,126,141,135,125,149,170,170,158,133,114,140,
#     145,150,178,163,172,178,199,199,184,162,146,166,
#     171,180,193,181,183,218,230,242,209,191,172,194,
#     196,196,236,235,229,243,264,272,237,211,180,201,
#     204,188,235,227,234,264,302,293,259,229,203,229,
#     242,233,267,269,270,315,364,347,312,274,237,278,
#     284,277,317,313,318,374,413,405,355,306,271,306,
#     315,301,356,348,355,422,465,467,404,347,305,336,
#     340,318,362,348,363,435,491,505,404,359,310,337,
#     360,342,406,396,420,472,548,559,463,407,362,405,
#     417,391,419,461,472,535,622,606,508,461,390,432
# ]
# ts = pd.DataFrame(
#     {"passengers": values},
#     index=pd.date_range(start="1949-01-01", periods=len(values), freq="MS")
# )
# ts.head()


## 3. Ursprüngliche Zeitreihe darstellen

Bevor STL angewendet wird, sollten die ursprünglichen Daten immer visuell geprüft werden.


In [ ]:
ax = ts["passengers"].plot(title="Monatliche internationale Flugpassagiere")
ax.set_xlabel("Datum")
ax.set_ylabel("Passagiere")
plt.show()


### Was bereits erkennbar ist

Die Reihe zeigt zwei wichtige Muster:

1. **Trend**: Die Passagierzahlen steigen im Allgemeinen im Laufe der Zeit.
2. **Saisonalität**: Es gibt jedes Jahr ein wiederkehrendes Muster.

Dadurch eignet sich der Datensatz gut für die STL-Zerlegung.


## 4. STL-Zerlegung anwenden

Da die Daten monatlich vorliegen und sich das saisonale Muster jedes Jahr wiederholt, beträgt die saisonale Periode:

\[
period = 12
\]

Das bedeutet, dass STL nach einem saisonalen Muster sucht, das sich alle 12 Monate wiederholt.


In [ ]:
stl = STL(ts["passengers"], period=12, robust=True)
result = stl.fit()

result


## 5. STL-Komponenten darstellen

STL trennt die ursprüngliche Zeitreihe in:

- beobachtete Daten
- Trend
- saisonale Komponente
- Residuenkomponente


In [ ]:
fig = result.plot()
fig.set_size_inches(12, 8)
plt.show()


## 6. Komponenten interpretieren

### Trend

Der Trend zeigt die langfristige Entwicklung der Passagiernachfrage.

In diesem Datensatz steigt der Trend von 1949 bis 1960 stark an. Das bedeutet, dass der Markt für Flugreisen im Laufe der Zeit wächst.

### Saisonale Komponente

Die saisonale Komponente zeigt das jährlich wiederkehrende Muster.

Einige Monate haben zum Beispiel regelmäßig höhere Passagierzahlen, während andere Monate regelmäßig niedrigere Passagierzahlen haben.

### Residuenkomponente

Die Residuenkomponente enthält das, was STL nicht durch Trend und Saisonalität erklären konnte.

Große Residuen können auf ungewöhnliche Ereignisse, besondere Umstände oder zufälliges Rauschen hinweisen.


## 7. Komponenten in einem DataFrame speichern

Für weitere Analysen ist es oft nützlich, die STL-Ausgabe in einer Tabelle zu speichern.


In [ ]:
components = pd.DataFrame({
    "beobachtet": ts["passengers"],
    "trend": result.trend,
    "saisonal": result.seasonal,
    "residuum": result.resid,
})

components.head(15)


## 8. Saisonales Muster nach Monat prüfen

Um das saisonale Verhalten leichter zu verstehen, können wir den durchschnittlichen saisonalen Effekt für jeden Monat berechnen.


In [ ]:
seasonality_by_month = components.copy()
month_names_de = {
    1: "Januar", 2: "Februar", 3: "März", 4: "April",
    5: "Mai", 6: "Juni", 7: "Juli", 8: "August",
    9: "September", 10: "Oktober", 11: "November", 12: "Dezember"
}
seasonality_by_month["month"] = seasonality_by_month.index.month.map(month_names_de)

monthly_seasonality = seasonality_by_month.groupby("month")["saisonal"].mean()
monthly_seasonality = monthly_seasonality.reindex([
    "Januar", "Februar", "März", "April", "Mai", "Juni",
    "Juli", "August", "September", "Oktober", "November", "Dezember"
])

monthly_seasonality


In [ ]:
ax = monthly_seasonality.plot(kind="bar", title="Durchschnittlicher saisonaler Effekt nach Monat")
ax.set_xlabel("Monat")
ax.set_ylabel("Saisonaler Effekt")
plt.xticks(rotation=45)
plt.show()


### Interpretation

Positive saisonale Werte bedeuten, dass der Monat normalerweise **mehr Passagiere hat, als allein durch den Trend zu erwarten wäre**.

Negative saisonale Werte bedeuten, dass der Monat normalerweise **weniger Passagiere hat, als allein durch den Trend zu erwarten wäre**.

Bei der Nachfrage im Flugverkehr zeigen Sommermonate häufig einen positiven saisonalen Effekt.


## 9. Ungewöhnliche Monate mithilfe von Residuen erkennen

Residuen zeigen Monate, die durch Trend und Saisonalität nicht gut erklärt werden.

Eine einfache Möglichkeit zur Identifikation ungewöhnlicher Monate besteht darin, Residuen zu markieren, die mehr als zwei Standardabweichungen von null entfernt sind.


In [ ]:
residual_std = components["residuum"].std()
threshold = 2 * residual_std

unusual_months = components[components["residuum"].abs() > threshold]
unusual_months


In [ ]:
ax = components["residuum"].plot(title="STL-Residuen mit Schwellenwert für ungewöhnliche Monate")
ax.axhline(threshold, linestyle="--")
ax.axhline(-threshold, linestyle="--")
ax.set_xlabel("Datum")
ax.set_ylabel("Residuum")
plt.show()


## 10. Warum STL für Prognosen nützlich ist

STL selbst ist kein Prognosemodell. Es ist eine Zerlegungsmethode.

Trotzdem hilft STL bei Prognosen, weil es die Zeitreihe in sinnvolle Bestandteile aufteilt:

- den Trend separat prognostizieren
- zukünftige saisonale Effekte schätzen
- Residuen als Rauschen behandeln oder separat modellieren

Ein typischer Prognose-Workflow ist:

1. STL verwenden, um die Saisonalität zu entfernen.
2. Die saisonbereinigte Reihe prognostizieren.
3. Die Saisonalität wieder hinzufügen, um die finale Prognose zu erhalten.

Dadurch wird das Prognoseproblem leichter verständlich und oft auch leichter modellierbar.


## 11. Einfache geschäftliche Schlussfolgerung

Für den Datensatz der Flugpassagiere zeigt STL:

- Die Passagiernachfrage ist über die Jahre stark gestiegen.
- Die Nachfrage weist eine klare jährliche Saisonalität auf.
- Einige Monate weichen vom normalen Verhalten ab, was in den Residuen sichtbar wird.

Für ein Unternehmen bedeutet das, dass die Planung nicht nur die rohen Passagierzahlen verwenden sollte.

Das Unternehmen sollte getrennt berücksichtigen:

- langfristiges Wachstum
- wiederkehrende saisonale Nachfrage
- ungewöhnliche einmalige Effekte

Genau deshalb ist STL in geschäftlichen Prognoseprojekten nützlich.


## 12. Mögliche Erweiterung für eine Aufgabe

Für eine Prognoseaufgabe könnten Sie vergleichen:

### Basisansatz

Verwenden Sie die Passagierzahl aus demselben Monat des Vorjahres als Prognose.

Beispiel:

\[
\hat{y}_t = y_{t-12}
\]

### Klassischer Ansatz

Verwenden Sie STL plus ein klassisches Prognosemodell wie exponentielle Glättung oder ARIMA.

### Deep-Learning-Ansatz

Verwenden Sie ein einfaches neuronales Netz, LSTM oder ein transformerbasiertes Modell für Zeitreihenprognosen.

Vergleichen Sie anschließend die Methoden anhand von Kennzahlen wie:

- MAE
- RMSE
- MAPE
